# Obstacles in C-space: notebook

The room of RO-009 (`images/roommap.py`) with the robot's disk footprint of radius 0.25 m and a 1.4 m x 0.3 m board. The geometry lives in `images/cspace.py`; here we check every number of the Note.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import shapely
from shapely.geometry import Point, LineString, Polygon
from scipy import ndimage
import cspace as c

## 1. The opening problem (Note, Section 1)

In [2]:
T = Polygon(c.TABLE)
corner = (3.5, 3.0)
d = T.distance(Point(*c.TEST))
print('centre', c.TEST, 'distance to the table:', round(d, 3), 'm; overlap of the disk:', round(c.R - d, 3), 'm')
print('doorway width:', c.DOOR[1] - c.DOOR[0], 'm; robot width:', 2 * c.R, 'm')

centre (3.5, 3.2) distance to the table: 0.2 m; overlap of the disk: 0.05 m
doorway width: 1.2000000000000002 m; robot width: 0.5 m


## 2. The disk robot's obstacle region (Note, Section 2)

In [3]:
D = c.cobs_disk()
for name, p in [('test', c.TEST), ('start', c.START[:2]), ('goal', c.GOAL)]:
    print(name, p, 'in the obstacle region:', D.intersects(Point(*p)))
print('doorway gap for the centre (numerical):', round(c.door_gap(D), 3), 'm; exact:', 1.2 - 2 * c.R, 'm')

test (3.5, 3.2) in the obstacle region: True
start (2.0, 1.0) in the obstacle region: False
goal (3.6, 4.0) in the obstacle region: False
doorway gap for the centre (numerical): 0.7 m; exact: 0.7 m


In [4]:
print('table area:', T.area, 'm2; perimeter:', round(T.length, 3), 'm')
print('turn at each corner (deg):', [round(t, 1) for t in c.turns(T)], 'sum', round(sum(c.turns(T)), 1))
steiner = T.area + T.length * c.R + np.pi * c.R ** 2
print('strips:', round(T.length * c.R, 3), 'corner wedges:', round(np.pi * c.R ** 2, 3))
print('grown area by the formula:', round(steiner, 3), 'm2; by shapely buffer:', round(T.buffer(c.R, quad_segs=256).area, 3), 'm2')

table area: 3.625 m2; perimeter: 7.704 m
turn at each corner (deg): [np.float64(104.0), np.float64(71.6), np.float64(94.4), np.float64(90.0)] sum 360.0
strips: 1.926 corner wedges: 0.196
grown area by the formula: 5.747 m2; by shapely buffer: 5.747 m2


## 3. The straight drive to the door (Note, Section 2.4)

In [5]:
L = LineString([c.START[:2], c.GOAL])
print('length:', L.length, 'm; heading:', round(np.degrees(np.arctan2(3, 1.6)), 1), 'deg')
print('closest approach to the table:', round(L.distance(T), 3), 'm; clearance of the disk:', round(L.distance(T) - c.R, 3), 'm')
print('clearance from the grown obstacles:', round(L.distance(D), 3), 'm')
t = ((1.5) * 1.6 + 2.0 * 3) / (1.6 ** 2 + 3 ** 2)
print('closest point on the drive:', round(2 + 1.6 * t, 3), round(1 + 3 * t, 3), 'after', round(t * L.length, 3), 'm')
others = {'pink pole': Point(1, 3).buffer(c.POLE_R), 'green pole': Point(6, 0.8).buffer(c.POLE_R)}
for k, g in others.items():
    print(k, round(L.distance(g), 3))
print('left door jamb (3.0, 4.0):', round(L.distance(Point(3.0, 4.0)), 3))

length: 3.4 m; heading: 61.9 deg
closest approach to the table: 0.382 m; clearance of the disk: 0.132 m
clearance from the grown obstacles: 0.132 m
closest point on the drive: 3.163 3.18 after 2.471 m
pink pole 1.724
green pole 3.524
left door jamb (3.0, 4.0): 0.529


## 4. The board: Minkowski sums (Note, Section 3)

In [6]:
a = (0.7, 0.0)                 # the middle of the board's front end, at heading 0
o = (3.0, 1.0)                 # the table's bottom-left corner
q = (o[0] - a[0], o[1] - a[1])
print('centre that puts the front end on the corner:', q)
C0 = c.cobs_board(0)
print('on the boundary of the C-obstacle slice:', round(C0.boundary.distance(Point(*q)), 9) == 0)
for deg in [0, 30, 45, 60, 90]:
    P = c.cobs_board_piece(T, np.radians(deg))
    print(deg, 'deg: table C-obstacle has', len(c.corners(P)), 'corners, area', round(P.area, 3), 'm2')

centre that puts the front end on the corner: (2.3, 1.0)
on the boundary of the C-obstacle slice: True
0 deg: table C-obstacle has 7 corners, area 7.595 m2
30 deg: table C-obstacle has 8 corners, area 8.394 m2
45 deg: table C-obstacle has 8 corners, area 8.358 m2
60 deg: table C-obstacle has 8 corners, area 8.028 m2
90 deg: table C-obstacle has 7 corners, area 8.145 m2


In [7]:
for deg in [0, 10, 18, 20, 30, 45, 60, 75, 90]:
    print(deg, 'deg: doorway gap', round(c.door_gap(c.cobs_board(np.radians(deg))), 3), 'm')
print('board width across the doorway at 0 deg:', c.BOARD[0], 'm; at 90 deg:', c.BOARD[1], 'm')

0 deg: doorway gap 0.0 m
10 deg: doorway gap 0.0 m
18 deg: doorway gap 0.002 m
20 deg: doorway gap 0.052 m
30 deg: doorway gap 0.219 m
45 deg: doorway gap 0.387 m
60 deg: doorway gap 0.547 m
75 deg: doorway gap 0.719 m
90 deg: doorway gap 0.9 m
board width across the doorway at 0 deg: 1.4 m; at 90 deg: 0.3 m


## 5. Connected components of free space (Note, Section 4)

In [8]:
xs, ys, Fd = c.free_grid(D, h=0.01)
lab, n = ndimage.label(Fd)
print('disk: pieces of free space in the room:', n)
for deg in [0, 60, 90]:
    xs, ys, F = c.free_grid(c.cobs_board(np.radians(deg)), h=0.01)
    lab, n = ndimage.label(F)
    areas = sorted([round((lab == i).sum() * 0.01 ** 2, 3) for i in range(1, n + 1)], reverse=True)
    print('board at', deg, 'deg:', n, 'pieces, areas (m2):', areas)
    if deg == 90:
        pick = lambda p: lab[int(round(p[1] / 0.01)), int(round(p[0] / 0.01))]
        print('  piece of start', pick(c.START[:2]), 'doorway', pick(c.GOAL), 'point (6.6, 1.3)', pick((6.6, 1.3)))

disk: pieces of free space in the room: 1


board at 0 deg: 2 pieces, areas (m2): [np.float64(10.441), np.float64(0.391)]


board at 60 deg: 4 pieces, areas (m2): [np.float64(6.258), np.float64(0.982), np.float64(0.152), np.float64(0.079)]
board at 90 deg: 3 pieces, areas (m2): [np.float64(7.124), np.float64(1.53), np.float64(0.004)]
  piece of start 1 doorway 1 point (6.6, 1.3) 2


In [9]:
# all headings at once: a 3D grid over (x, y, heading), 0.01 m and 1 degree; heading wraps (the board repeats every 180 deg)
h, ths = 0.01, np.arange(0, 180, 1.0)
free = np.stack([c.free_grid(c.cobs_board(np.radians(t)), h=h)[2] for t in ths])
lab, n = ndimage.label(free)
parent = list(range(n + 1))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
m = (lab[0] > 0) & (lab[-1] > 0)        # slice 179 deg meets slice 180 deg = 0 deg
for u, v in set(zip(lab[0][m].tolist(), lab[-1][m].tolist())):
    parent[find(u)] = find(v)
print('pieces of free space in (x, y, heading):', len({find(i) for i in range(1, n + 1)}))

pieces of free space in (x, y, heading): 1


## 6. Turning on the spot at the start (Note, Section 4)

In [10]:
print('board free at (2, 1) for every heading 30 to 90 deg:',
      all(not c.cobs_board(np.radians(t)).intersects(Point(2, 1)) for t in range(30, 91)))

board free at (2, 1) for every heading 30 to 90 deg: True
